<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

## Understand the KKT Conditions

**See when a bound blocks a better clock setting, then check whether a candidate respects that bound.**

The clock observations, squared-error objective, and exact calibration stay fixed. An upper bound on the initial correction is added. KKT connects that bound to the same Lagrangian idea.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 1 · Add an upper bound to the calibrated clock

The recorded errors are $-2,-1,-1,0$ min at $0,1,2,3$ h. Choose correction $q$ in min and rate $r$ in min/h. The remaining errors are $y_i=d_i+q+rt_i$ for $i=1,\ldots,4$, and the score is $\phi(q,r)=\sum_{i=1}^{4}y_i^2$ in min².

Calibration at one hour still requires $q+(1\,\mathrm h)r=1\,\mathrm{min}$. Using numerical values in these units, write the two requirements as

$$
h=q+r-1=0,\qquad g=q-q_{\max}\le0.
$$

Here $q_{\max}$ is the allowed upper correction, in min. With $q_{\max}=1$, the equality-only minimum $(1.5,-0.5)$ is rejected. With $q_{\max}=2$, it is allowed. Changing the bound changes feasibility; it does not change a fixed setting's clock errors.

The code evaluates the same physical errors and checks both requirements.


In [ ]:
import numpy as np

# Fixed observations and their elapsed times
OBSERVED_ERRORS = np.array([-2.0, -1.0, -1.0, 0.0])  # min
OBSERVATION_TIMES = np.array([0.0, 1.0, 2.0, 3.0])   # h

# Calibration requirement: correction at 1 h must equal 1 min
CALIBRATION_TIME = 1.0        # h
REQUIRED_CORRECTION = 1.0     # min
CHECK_TOLERANCE = 1e-9        # numerical comparison setting


def simulate_clock(decision):
    correction, rate = np.asarray(decision, dtype=float)
    return OBSERVED_ERRORS + correction + rate * OBSERVATION_TIMES


def evaluate_clock(decision):
    decision = np.asarray(decision, dtype=float)
    remaining_errors = simulate_clock(decision)
    correction, rate = decision
    equality_residual = correction + CALIBRATION_TIME * rate - REQUIRED_CORRECTION
    gradient = 2 * np.array([np.sum(remaining_errors),
                             np.dot(OBSERVATION_TIMES, remaining_errors)])
    return {
        "decision": decision,
        "response": remaining_errors,
        "objective": float(np.sum(remaining_errors ** 2)),
        "gradient": gradient,
        "equality_residual": float(equality_residual),
        "feasible": bool(abs(equality_residual) <= CHECK_TOLERANCE),
    }


CORRECTION_LIMIT = 1.0  # min


def evaluate_bounded_clock(decision, correction_limit=CORRECTION_LIMIT):
    result = evaluate_clock(decision)
    inequality_residual = float(result["decision"][0] - correction_limit)
    result.update({
        "inequality_residual": inequality_residual,
        "feasible": bool(result["feasible"] and inequality_residual <= CHECK_TOLERANCE),
    })
    return result


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 2 · Distinguish the Lagrangian from KKT

The **Lagrangian is a function**. The **Karush–Kuhn–Tucker (KKT) conditions are checks on settings and multipliers**, expressed using that function. KKT is not a replacement objective or a search algorithm.

Keep $\nu$ for the equality and use $\mu$ for the inequality:

$$
L(q,r,\nu,\mu)=\phi(q,r)+\nu h(q,r)+\mu g(q,r).
$$

| Concept | Meaning in this example |
|:---|:---|
| Lagrangian | Combines the score and constraint residuals for the slope calculation |
| Equality multiplier method | Balances slopes while keeping $h=0$ |
| KKT conditions | Also account for the inequality's direction and whether it is active |

With no inequality, the KKT conditions reduce to the equality multiplier conditions. With an inequality, its multiplier cannot be treated exactly like an equality multiplier. For minimization with $g\le0$, the sign convention is $\mu\ge0$; $\nu$ remains unrestricted. Both multipliers are solved, not chosen as clock settings or preference weights.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 3 · See whether the bound blocks improvement

A bound is **active** when $q=q_{\max}$ and **inactive** when $q<q_{\max}$. Keep calibration and compare three limits:

| Upper bound (min) | Best settings $(q,r)$ | Bound status | $\mu$ |
|:---|:---|:---|---:|
| $1$ | $(1,0)$ | Active: prevents moving toward the equality-only minimum | $6$ |
| $1.5$ | $(1.5,-0.5)$ | Active: the equality-only minimum is exactly on the bound | $0$ |
| $2$ | $(1.5,-0.5)$ | Inactive: the equality-only minimum is inside the limit | $0$ |

The figure changes only the upper bound. The orange point is the selected setting; the teal part of each line is feasible in the displayed range. The rate always adjusts to $r=1-q$.
<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-1_constrained_optimization_kkt/assets/bound_activity_and_multiplier.png" alt="Three correction limits retain exact calibration. At limit 1, the selected correction 1 is on the boundary with inequality multiplier 6. At limit 1.5, the selected correction 1.5 is on the boundary with multiplier zero. At limit 2, correction 1.5 is inside the boundary with multiplier zero." width="512" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: #fff;">
</div>


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

At an inactive bound, there is room for sufficiently small moves on either side, so that bound supplies no slope contribution at a KKT point. Its multiplier is zero. The condition

$$
\mu g=0
$$

is called **complementarity**. Either the bound has room left ($g<0$, forcing $\mu=0$), or it is active ($g=0$, allowing $\mu\ge0$). The middle row shows that an active bound can still have a zero multiplier.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 4 · Understand the multiplier sign through a small move

Under the tight limit $q_{\max}=1$, moving from $(1,0)$ to $(1.1,-0.1)$ would lower the score from 2 to 1.46 min², but the bound rejects it. Moving the other way to $(0.9,0.1)$ is feasible and raises the score to 2.66 min². The positive multiplier $\mu=6$ reflects a bound blocking the improving direction.

Now hold the loose limit $q_{\max}=2$ fixed and consider the boundary setting $(2,-1)$. Moving inward to $(1.9,-0.9)$ is feasible and lowers the score from 2 to 1.46 min². This boundary setting cannot be a minimum. Balancing its slopes would require $\mu=-6$, which fails the required sign.

The sign follows the written inequality convention. Along the calibration line, stationarity reads “score slope + $\mu=0$.” At a minimizing upper boundary, moving left must not lower the score, so that slope is nonpositive and $\mu$ is nonnegative. Here the slopes at $q=1$ and $q=2$ are $-6$ and $+6$, respectively.

The code compares these moves directly. Infeasible settings are labeled as rejected, regardless of score.


In [ ]:
for limit, corrections in [(1.0, (0.9, 1.0, 1.1)), (2.0, (1.9, 2.0))]:
    print(f"Upper bound = {limit:.1f} min")
    for correction in corrections:
        result = evaluate_bounded_clock([correction, 1 - correction], limit)
        status = "feasible" if result["feasible"] else "REJECT"
        print(f"  q = {correction:.1f}, r = {1 - correction:.1f}: "
              f"{status}, score = {result['objective']:.2f} min²")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 5 · Run the four checks together

| Check | Compact expression | Plain meaning |
|:---|:---|:---|
| Feasibility | $h=0$, $g\le0$ | Both requirements hold |
| Multiplier sign | $\mu\ge0$ | The inequality contribution has the correct direction |
| Complementarity | $\mu g=0$ | An inactive bound contributes zero |
| Stationarity | $\partial L/\partial q=0$, $\partial L/\partial r=0$ | The objective and constraint slopes balance |

For the tight-limit candidate $(1,0)$, the objective slopes are $(0,6)$. The equality multiplier $\nu=-6$ adds $(-6,-6)$, and $\mu=6$ adds $(6,0)$. Their componentwise sum is $(0,0)$: stationarity holds.

The checker reports each condition separately. The first two cases below pass all four. A negative inequality multiplier fails the sign check; keeping a positive multiplier on a loose bound fails complementarity. The equality-only answer under the tight bound fails feasibility. Balanced slopes alone cannot detect all these errors.


In [ ]:
def check_candidate(decision, equality_multiplier, inequality_multiplier,
                    correction_limit=CORRECTION_LIMIT):
    result = evaluate_bounded_clock(decision, correction_limit)
    stationarity = (result["gradient"]
                    + equality_multiplier * np.array([1.0, CALIBRATION_TIME])
                    + inequality_multiplier * np.array([1.0, 0.0]))
    checks = {
        "feasibility": result["feasible"],
        "sign": bool(inequality_multiplier >= -CHECK_TOLERANCE),
        "complementarity": bool(abs(inequality_multiplier * result["inequality_residual"])
                                <= CHECK_TOLERANCE),
        "stationarity": bool(np.all(np.abs(stationarity) <= CHECK_TOLERANCE)),
    }
    return {"evaluation": result, "checks": checks,
            "stationarity_residual": stationarity,
            "all_pass": all(checks.values())}


In [ ]:
cases = [
    ("Tight-bound minimum", 1.0, [1.0, 0.0], -6.0, 6.0),
    ("Loose-bound minimum", 2.0, [1.5, -0.5], 2.0, 0.0),
    ("Wrong multiplier sign", 2.0, [2.0, -1.0], 10.0, -6.0),
    ("Multiplier on loose bound", 2.0, [1.0, 0.0], -6.0, 6.0),
    ("Setting outside bound", 1.0, [1.5, -0.5], 2.0, 0.0),
]
print(f"{'Case':27} {'Feasible':9} {'Sign':9} {'Complement':11} {'Stationary':10}")
for name, limit, decision, equality_multiplier, inequality_multiplier in cases:
    checked = check_candidate(decision, equality_multiplier, inequality_multiplier, limit)
    marks = ["PASS" if passed else "FAIL" for passed in checked["checks"].values()]
    print(f"{name:27} {marks[0]:9} {marks[1]:9} {marks[2]:11} {marks[3]:10}")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 6 · Interpret a passing result

KKT is a necessary check for smooth constrained minima under suitable regularity assumptions. Those assumptions hold here: the equality and active-bound slope vectors, $(1,1)$ and $(1,0)$, are independent.

For this clock problem, the objective is a convex quadratic and the constraints are affine. In this setting, passing KKT is also sufficient for a global minimum; strict convexity makes it unique. With a general nonconvex objective, a KKT point can fail to be a minimum. Feasibility alone and stationarity alone are each weaker than the complete check.

The terminology follows [Boyd and Vandenberghe, Sections 5.1 and 5.5](https://web.stanford.edu/~boyd/cvxbook/bv_cvxbook.pdf).
